### MINIMIZACAO EICONAL ATE t = 0.1 COM 3 PARAMETROS LIVRES

## codigo que calula dif sigma eik sem minmizacao para 3 pontos de q2 com otimizacao

In [14]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad, quad

In [15]:
# Load experimental data
atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_508878/939602129.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../data/data_0_1/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)


In [16]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

In [17]:
n_points = 8000 # Number of points for fixed_quad integration


q_max_chi = 5.0          # limite de q na Eq. 23
b_max = 15.0
abs_t = 0.1


eps_rel = 1e-6
eps_abs = 1e-12


limit = 10000

In [18]:
sqrt_s = 7000

s = sqrt_s**2

eps_eik = 0.09583
mg_eik = 0.93
a1_eik = 1.4	

mg_born = 0.421
eps_born = 0.0753
a1_born = 1.517

In [19]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1):
    return np.exp(-(a1 * q2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q2, phi, mg, a1, m2_func):
    qk_cos = np.sqrt(q2) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1)
    G_minus = G_p(factor, a1)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, m2_func) - T_2(k, q_val, phi, mg, a1, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  



In [20]:
import numpy as np
from functools import lru_cache

@lru_cache(maxsize=8)
def _get_gauss_legendre_nodes(n_points):
    """Nos e pesos de Gauss-Legendre em [0,1], cacheados (evita recalculo caro a cada chamada)."""
    nodes, weights = np.polynomial.legendre.leggauss(n_points)
    x_nodes = 0.5 * (nodes + 1.0)  # mapeado de [-1,1] para [0,1]
    return x_nodes, weights

def full_int(mg, a1, m2_func, q_val, sqrt_s, n_points=n_points):
    """
    Versao vetorizada de full_int. Reproduz EXATAMENTE o mesmo calculo
    numerico da versao original (mesmo pareamento de nos x_i <-> y_i
    que o fixed_quad aninhado original produzia), apenas sem o overhead
    de reconstruir nos/pesos e chamar fixed_quad/lambdas repetidamente.

    Validado numericamente: diferenca vs versao original ~1e-10 a 1e-19
    (ruido de ponto flutuante, nao erro de metodo).
    """
    q_val = np.atleast_1d(q_val)
    x_nodes, weights = _get_gauss_legendre_nodes(n_points)

    k = sqrt_s * x_nodes
    phi = 2 * np.pi * x_nodes
    jacobian = 2 * np.pi * sqrt_s

    results = []
    for q in q_val:
        vals = k * (
            T_1(k, q, phi, mg, a1, m2_func) - T_2(k, q, phi, mg, a1, m2_func)
        ) * jacobian
        # fixed_quad interno: (b-a)/2 * sum(w * vals), com b-a=1
        integral_value = 0.5 * np.sum(weights * vals)
        results.append(integral_value)

    return np.array(results) if len(results) > 1 else results[0]

In [21]:
# for q2 in [0, 0.1, 0.2]:
#     t = -q2

#     integral_value = full_int(mg_born, a1_born, m2_pl, q2, sqrt_s)

#     amp_value = amp_calculation(integral_value, s, eps_born, t)

#     diff_cross_section = (amp_value.imag * amp_value.imag) / (16 * np.pi * s ** 2) * 0.389379323

#     # print(f"q2 = {q2}, diff_cross_section = {diff_cross_section:.6e} mb/GeV^2")
#     print(f"integral_value = {integral_value:.6e}")

In [22]:
# # # ── Eq. 23: χ(s,b) como integral direta em q ─────────────────────────────────
# def chi(b_val, mg, a1, eps, m2_func, sqrt_s):

#     s_local = sqrt_s ** 2  # [CORREÇÃO 1] s local, não captura variável global

#     def integrand(q_val):
#         """Integrando complexo — full_int chamado uma única vez por ponto."""
#         # [CORREÇÃO 2] integrando único complexo: evita chamar full_int 2x
#         q2_val    = q_val ** 2
#         t         = -q2_val
#         diff_t    = full_int(mg, a1, m2_func, q2_val, sqrt_s)
#         born_amp  = amp_calculation(diff_t, s_local, eps, t)
#         return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

#     # ── Soma de Riemann (regra do ponto médio) ──────────────────────────────
#     dq = q_max_chi / n_points
#     q_mid = (np.arange(n_points) + 0.5) * dq

#     result = 0.0 + 0.0j
#     for i, q_val in enumerate(q_mid):
#         val = integrand(q_val) * dq
#         result += val
#     print(f"b = {b_val}, chi = {result}")

#     return result


# # chi(0.0, mg_eik, a1_eik, eps_eik, m2_pl, 7000)

In [23]:
from functools import lru_cache

def chi(b_val, mg, a1, eps, m2_func, sqrt_s):

    s_local = sqrt_s ** 2

    @lru_cache(maxsize=None)
    def _integrand_complex(q_val):
        """Calculo pesado (full_int + amp_calculation) cacheado por q_val.
        Evita recalcular quando o mesmo q_val e amostrado tanto na
        integracao da parte real quanto na da parte imaginaria pelo quad."""
        q2_val = q_val ** 2
        t = -q2_val
        diff_t = full_int(mg, a1, m2_func, q2_val, sqrt_s)
        born_amp = amp_calculation(diff_t, s_local, eps, t)
        return (1.0 / s_local) * q_val * j0(b_val * q_val) * born_amp

    def integrand_real(q_val):
        return _integrand_complex(q_val).real

    def integrand_imag(q_val):
        return _integrand_complex(q_val).imag

    real_part, _ = quad(integrand_real, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(integrand_imag, 0, q_max_chi, epsrel=eps_rel, epsabs=eps_abs, limit=limit)

    return real_part + 1j * imag_part

In [24]:
# lst_b_integration = [
#     0.000234375, 0.152109375
# ]


# for b_val in lst_b_integration:
#     chi_value = chi(b_val, mg_eik, a1_eik, eps_eik, m2_pl, 7000 )
#     print(f"b = {b_val}, chi = {chi_value}")

In [25]:
# for n in [1000, 2000, 4000, 8000, 16000, 32000]:
#     n_points = n
#     val = chi(0, mg_eik, a1_eik, eps_eik, m2_pl, sqrt_s)
#     print(f"n_points={n}: chi = {val}")

In [26]:
# ── Eq. 24: A_eik(s,t) — integral em b com χ(b) calculado internamente ───────

lst_q2 = np.linspace(0, 0.1, 10)


for q2_exp in [0.011, 0.0307, 0.0959]:

    q_exp = np.sqrt(q2_exp)

    def integrand(b_val):
        chi_val = chi(b_val, mg_eik, a1_eik, eps_eik, m2_pl, sqrt_s)
        return b_val * j0(q_exp * b_val) * (1 - np.exp(1j * chi_val))

    real_part, _ = quad(lambda b: np.real(integrand(b)), 0, b_max, epsrel=eps_rel, epsabs=eps_abs, limit=limit)
    imag_part, _ = quad(lambda b: np.imag(integrand(b)), 0, b_max, epsrel=eps_rel, epsabs=eps_abs, limit=limit)

    integral_b = real_part + 1j * imag_part

    amp_eik = 1j * s * integral_b

    diff_sigma_eik = (amp_eik.imag * amp_eik.imag) * (np.pi/s**2) * 0.389379323

    print(50*"=")
    print(f"  q2 = {q2_exp}, A_eik = {amp_eik}")
    print(f"q2 = {q2_exp}, diff_sigma_eik = {diff_sigma_eik}")

  q2 = 0.011, A_eik = 848075905.8322809j
q2 = 0.011, diff_sigma_eik = 366.437615270507
  q2 = 0.0307, A_eik = 704506975.8975058j
q2 = 0.0307, diff_sigma_eik = 252.87226225117564
  q2 = 0.0959, A_eik = 370112891.94571626j
q2 = 0.0959, diff_sigma_eik = 69.79093620002227
